# Clustering Demonstration Notebook

This notebook supports a 3-hour clustering demonstration session using public datasets.

It covers:

- Loading public datasets
- Data cleaning and preprocessing
- Feature scaling
- K-Means clustering from scratch
- Elbow method from scratch
- Silhouette score from scratch
- Agglomerative hierarchical clustering from scratch
- Cluster profiling and interpretation
- Optional visualizations when `matplotlib` is installed

Public datasets used:

- Palmer Penguins dataset: `https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv`
- UCI Wine dataset: `https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data`

## 1. Imports and Utility Setup

The main implementation uses only `numpy` and `pandas`.

`matplotlib` is optional. If it is not installed, the notebook will still run and print tabular results.

In [ ]:
import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib available:", plt is not None)

## 2. Load Public Datasets

We will use two public datasets:

1. **Palmer Penguins** for an intuitive segmentation example.
2. **UCI Wine** for a classic numeric clustering benchmark.

The true labels are loaded only for later interpretation. The clustering algorithms will not use them during training.

In [ ]:
penguins_url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
wine_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data"

penguins = pd.read_csv(penguins_url)

wine_columns = [
    "class_label",
    "alcohol",
    "malic_acid",
    "ash",
    "alcalinity_of_ash",
    "magnesium",
    "total_phenols",
    "flavanoids",
    "nonflavanoid_phenols",
    "proanthocyanins",
    "color_intensity",
    "hue",
    "od280_od315",
    "proline",
]
wine = pd.read_csv(wine_url, header=None, names=wine_columns)

print("Penguins shape:", penguins.shape)
display(penguins.head())

print("Wine shape:", wine.shape)
display(wine.head())

## 3. Preprocess the Penguins Dataset

We will cluster penguins using only numeric body measurements:

- `bill_length_mm`
- `bill_depth_mm`
- `flipper_length_mm`
- `body_mass_g`

The `species` column will not be used for training. We keep it only to check whether discovered clusters align with real species.

In [ ]:
penguin_features = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
]

penguins_clean = penguins.dropna(subset=penguin_features + ["species"]).copy()
X_penguins = penguins_clean[penguin_features].to_numpy(dtype=float)
y_penguins = penguins_clean["species"].to_numpy()

print("Rows after cleaning:", len(penguins_clean))
display(penguins_clean[penguin_features + ["species"]].head())

## 4. Scaling From Scratch

Clustering is distance-based. If one feature has a much larger scale than another, it can dominate the distance calculation.

For example, `body_mass_g` has values in thousands while `bill_depth_mm` is usually below 25. Standardization puts all features on comparable scale.

In [ ]:
class StandardScalerFromScratch:
    def __init__(self):
        self.mean_ = None
        self.std_ = None

    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        self.std_[self.std_ == 0] = 1
        return self

    def transform(self, X):
        return (X - self.mean_) / self.std_

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)


scaler = StandardScalerFromScratch()
X_penguins_scaled = scaler.fit_transform(X_penguins)

print("Original feature means:")
print(pd.Series(X_penguins.mean(axis=0), index=penguin_features))

print("\nScaled feature means, approximately zero:")
print(pd.Series(X_penguins_scaled.mean(axis=0), index=penguin_features))

print("\nScaled feature standard deviations, approximately one:")
print(pd.Series(X_penguins_scaled.std(axis=0), index=penguin_features))

## 5. K-Means Clustering From Scratch

K-Means alternates between two steps:

1. Assign every point to the nearest centroid.
2. Move each centroid to the mean of the points assigned to it.

It repeats until the centroids stop moving or the maximum number of iterations is reached.

In [ ]:
class KMeansFromScratch:
    def __init__(self, k=3, max_iter=100, tol=1e-4, random_state=42):
        self.k = k
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self.centroids = None
        self.labels = None
        self.inertia_ = None
        self.n_iter_ = 0

    def _euclidean_distance(self, X, centroids):
        return np.sqrt(((X[:, np.newaxis, :] - centroids[np.newaxis, :, :]) ** 2).sum(axis=2))

    def fit(self, X):
        rng = np.random.default_rng(self.random_state)
        random_indices = rng.choice(X.shape[0], self.k, replace=False)
        self.centroids = X[random_indices].copy()

        for iteration in range(1, self.max_iter + 1):
            old_centroids = self.centroids.copy()

            distances = self._euclidean_distance(X, self.centroids)
            self.labels = np.argmin(distances, axis=1)

            new_centroids = []
            for cluster_id in range(self.k):
                cluster_points = X[self.labels == cluster_id]

                if len(cluster_points) == 0:
                    replacement_index = rng.integers(0, X.shape[0])
                    new_centroids.append(X[replacement_index])
                else:
                    new_centroids.append(cluster_points.mean(axis=0))

            self.centroids = np.array(new_centroids)
            centroid_shift = np.sqrt(((self.centroids - old_centroids) ** 2).sum())

            if centroid_shift < self.tol:
                self.n_iter_ = iteration
                break
        else:
            self.n_iter_ = self.max_iter

        final_distances = self._euclidean_distance(X, self.centroids)
        closest_distances = final_distances[np.arange(X.shape[0]), self.labels]
        self.inertia_ = np.sum(closest_distances ** 2)
        return self

    def predict(self, X):
        distances = self._euclidean_distance(X, self.centroids)
        return np.argmin(distances, axis=1)

## 6. Run K-Means on Penguins

We will start with `K = 3` because the penguins dataset has three known species. In a real unsupervised problem, the species labels would not be available.

In [ ]:
kmeans_penguins = KMeansFromScratch(k=3, random_state=7)
kmeans_penguins.fit(X_penguins_scaled)

penguins_result = penguins_clean.copy()
penguins_result["kmeans_cluster"] = kmeans_penguins.labels

print("Iterations:", kmeans_penguins.n_iter_)
print("Inertia:", kmeans_penguins.inertia_)
display(penguins_result[["species", "kmeans_cluster"] + penguin_features].head())

print("\nCluster counts:")
display(penguins_result["kmeans_cluster"].value_counts().sort_index())

print("\nCluster vs actual species, for interpretation only:")
display(pd.crosstab(penguins_result["kmeans_cluster"], penguins_result["species"]))

## 7. Cluster Profiling

Cluster labels such as `0`, `1`, and `2` are not meaningful by themselves. We profile each cluster to understand what it represents.

In [ ]:
penguin_profile = penguins_result.groupby("kmeans_cluster")[penguin_features].mean().round(2)
penguin_profile["count"] = penguins_result.groupby("kmeans_cluster").size()
display(penguin_profile)

print("Example interpretation:")
for cluster_id, row in penguin_profile.iterrows():
    print(
        f"Cluster {cluster_id}: avg flipper length={row['flipper_length_mm']}, "
        f"avg body mass={row['body_mass_g']}, count={int(row['count'])}"
    )

## 8. Optional 2D Visualization Using PCA From Scratch

The dataset has four numeric features. To visualize it in two dimensions, we can use a simple PCA implementation from scratch.

In [ ]:
def pca_2d_from_scratch(X):
    X_centered = X - X.mean(axis=0)
    covariance_matrix = np.cov(X_centered, rowvar=False)
    eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)
    sorted_indices = np.argsort(eigenvalues)[::-1]
    top_vectors = eigenvectors[:, sorted_indices[:2]]
    return X_centered @ top_vectors


X_penguins_2d = pca_2d_from_scratch(X_penguins_scaled)

if plt is not None:
    plt.figure(figsize=(8, 5))
    plt.scatter(
        X_penguins_2d[:, 0],
        X_penguins_2d[:, 1],
        c=kmeans_penguins.labels,
        cmap="viridis",
        alpha=0.8,
    )
    plt.title("K-Means Clusters on Palmer Penguins, PCA 2D View")
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.show()
else:
    preview = pd.DataFrame({
        "pc1": X_penguins_2d[:, 0],
        "pc2": X_penguins_2d[:, 1],
        "cluster": kmeans_penguins.labels,
        "species": y_penguins,
    })
    display(preview.head(10))
    print("Install matplotlib to view the scatter plot.")

## 9. Elbow Method From Scratch

The elbow method checks how inertia changes as K increases.

Inertia usually decreases as K increases, but after a certain point the improvement becomes smaller. That bend is called the elbow.

In [ ]:
k_values = range(1, 9)
inertia_values = []

for k in k_values:
    model = KMeansFromScratch(k=k, random_state=7)
    model.fit(X_penguins_scaled)
    inertia_values.append(model.inertia_)

elbow_table = pd.DataFrame({
    "k": list(k_values),
    "inertia": inertia_values,
})
display(elbow_table)

if plt is not None:
    plt.figure(figsize=(7, 4))
    plt.plot(elbow_table["k"], elbow_table["inertia"], marker="o")
    plt.title("Elbow Method on Penguins Dataset")
    plt.xlabel("Number of clusters K")
    plt.ylabel("Inertia")
    plt.show()
else:
    print("Install matplotlib to view the elbow chart.")

## 10. Silhouette Score From Scratch

Silhouette score checks whether points are closer to their own cluster than to the nearest other cluster.

- Close to `1`: strong cluster assignment
- Around `0`: point lies near a boundary
- Negative: point may be assigned to the wrong cluster

In [ ]:
def pairwise_distances(X):
    n = X.shape[0]
    distances = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            distances[i, j] = np.sqrt(np.sum((X[i] - X[j]) ** 2))

    return distances


def silhouette_score_from_scratch(X, labels):
    distances = pairwise_distances(X)
    unique_labels = np.unique(labels)
    scores = []

    for i in range(X.shape[0]):
        own_cluster = labels[i]

        same_cluster_indices = np.where(labels == own_cluster)[0]
        same_cluster_indices = same_cluster_indices[same_cluster_indices != i]

        if len(same_cluster_indices) == 0:
            a = 0
        else:
            a = distances[i, same_cluster_indices].mean()

        b_values = []
        for other_cluster in unique_labels:
            if other_cluster == own_cluster:
                continue

            other_cluster_indices = np.where(labels == other_cluster)[0]
            b_values.append(distances[i, other_cluster_indices].mean())

        b = min(b_values)
        scores.append((b - a) / max(a, b))

    return float(np.mean(scores))


silhouette_rows = []

for k in range(2, 9):
    model = KMeansFromScratch(k=k, random_state=7)
    model.fit(X_penguins_scaled)
    silhouette_rows.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette_score": silhouette_score_from_scratch(X_penguins_scaled, model.labels),
    })

silhouette_table = pd.DataFrame(silhouette_rows)
display(silhouette_table)

## 11. Hierarchical Clustering From Scratch

Agglomerative hierarchical clustering starts with each point as its own cluster. It repeatedly merges the closest clusters until the desired number of clusters remains.

This from-scratch implementation supports:

- `single` linkage
- `complete` linkage
- `average` linkage

To keep the demonstration fast, we will run it on a deterministic sample of the penguins data.

In [ ]:
class AgglomerativeClusteringFromScratch:
    def __init__(self, n_clusters=3, linkage="average"):
        self.n_clusters = n_clusters
        self.linkage = linkage
        self.labels = None
        self.merge_history = []

    def _euclidean_distance(self, point_a, point_b):
        return np.sqrt(np.sum((point_a - point_b) ** 2))

    def _cluster_distance(self, X, cluster_a, cluster_b):
        distances = []

        for index_a in cluster_a:
            for index_b in cluster_b:
                distances.append(self._euclidean_distance(X[index_a], X[index_b]))

        distances = np.array(distances)

        if self.linkage == "single":
            return distances.min()

        if self.linkage == "complete":
            return distances.max()

        if self.linkage == "average":
            return distances.mean()

        raise ValueError("linkage must be one of: single, complete, average")

    def fit(self, X):
        clusters = [[i] for i in range(X.shape[0])]

        while len(clusters) > self.n_clusters:
            min_distance = float("inf")
            clusters_to_merge = (None, None)

            for i in range(len(clusters)):
                for j in range(i + 1, len(clusters)):
                    distance = self._cluster_distance(X, clusters[i], clusters[j])

                    if distance < min_distance:
                        min_distance = distance
                        clusters_to_merge = (i, j)

            i, j = clusters_to_merge
            new_cluster = clusters[i] + clusters[j]
            self.merge_history.append((clusters[i], clusters[j], min_distance))

            clusters.pop(j)
            clusters.pop(i)
            clusters.append(new_cluster)

        labels = np.empty(X.shape[0], dtype=int)

        for cluster_id, cluster in enumerate(clusters):
            for data_index in cluster:
                labels[data_index] = cluster_id

        self.labels = labels
        return self

In [ ]:
sample_size = 90
sample = penguins_clean.sample(n=sample_size, random_state=42).copy()
X_sample = sample[penguin_features].to_numpy(dtype=float)
X_sample_scaled = StandardScalerFromScratch().fit_transform(X_sample)

hierarchical = AgglomerativeClusteringFromScratch(n_clusters=3, linkage="average")
hierarchical.fit(X_sample_scaled)

sample["hierarchical_cluster"] = hierarchical.labels

print("Number of merge operations:", len(hierarchical.merge_history))
print("First five merge operations:")
for merge in hierarchical.merge_history[:5]:
    print(merge)

print("\nCluster counts:")
display(sample["hierarchical_cluster"].value_counts().sort_index())

print("\nCluster vs actual species, for interpretation only:")
display(pd.crosstab(sample["hierarchical_cluster"], sample["species"]))

## 12. Linkage Comparison

Different linkage methods can produce different clusters.

In [ ]:
linkage_results = []

for linkage in ["single", "complete", "average"]:
    model = AgglomerativeClusteringFromScratch(n_clusters=3, linkage=linkage)
    model.fit(X_sample_scaled)
    score = silhouette_score_from_scratch(X_sample_scaled, model.labels)
    linkage_results.append({
        "linkage": linkage,
        "silhouette_score": score,
        "cluster_counts": dict(pd.Series(model.labels).value_counts().sort_index()),
    })

linkage_table = pd.DataFrame(linkage_results)
display(linkage_table)

## 13. Second Public Dataset: UCI Wine

The Wine dataset contains chemical measurements for wines from three cultivars. We will cluster the wines without using the true cultivar label.

In [ ]:
wine_features = [col for col in wine.columns if col != "class_label"]
X_wine = wine[wine_features].to_numpy(dtype=float)
y_wine = wine["class_label"].to_numpy()

X_wine_scaled = StandardScalerFromScratch().fit_transform(X_wine)

wine_kmeans = KMeansFromScratch(k=3, random_state=11)
wine_kmeans.fit(X_wine_scaled)

wine_result = wine.copy()
wine_result["cluster"] = wine_kmeans.labels

print("Wine K-Means inertia:", wine_kmeans.inertia_)
print("Wine K-Means silhouette:", silhouette_score_from_scratch(X_wine_scaled, wine_kmeans.labels))

print("\nCluster vs actual wine class, for interpretation only:")
display(pd.crosstab(wine_result["cluster"], wine_result["class_label"]))

wine_profile = wine_result.groupby("cluster")[wine_features].mean().round(2)
wine_profile["count"] = wine_result.groupby("cluster").size()
display(wine_profile)

## 14. Optional Wine Visualization

In [ ]:
X_wine_2d = pca_2d_from_scratch(X_wine_scaled)

if plt is not None:
    plt.figure(figsize=(8, 5))
    plt.scatter(
        X_wine_2d[:, 0],
        X_wine_2d[:, 1],
        c=wine_kmeans.labels,
        cmap="viridis",
        alpha=0.8,
    )
    plt.title("K-Means Clusters on UCI Wine Dataset, PCA 2D View")
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.show()
else:
    wine_preview = pd.DataFrame({
        "pc1": X_wine_2d[:, 0],
        "pc2": X_wine_2d[:, 1],
        "cluster": wine_kmeans.labels,
        "actual_class_for_interpretation": y_wine,
    })
    display(wine_preview.head(10))
    print("Install matplotlib to view the scatter plot.")

## 15. Final Teaching Notes

### What to emphasize during the session

- Clustering is unsupervised; labels are not used during training.
- Scaling is essential for distance-based algorithms.
- K-Means is fast and useful for large numerical segmentation problems.
- Hierarchical clustering is slower but more interpretable for smaller datasets.
- Cluster evaluation requires both metrics and business judgment.
- Cluster IDs are arbitrary; profiling gives them meaning.

### Suggested discussion questions

- Did K-Means recover the actual species or wine classes perfectly?
- Why might clustering differ from known labels?
- Which K would you choose from the elbow and silhouette tables?
- Which linkage method performed best on the sample?
- How would you convert these clusters into business actions?